In [ ]:
import os
import base64
import pprint
import requests
import pandas as pd
import time
import base64
import numpy as np
import json
import re

from google import genai
from google.colab import userdata
from google.genai import types

In [ ]:
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY_3")

In [ ]:
MODEL_2 = "gemma-3-27b-it"
CONFIG = types.GenerateContentConfig(
    temperature=0.1
)

# Load dataset

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}

# Load original set
train_original_df = pd.read_parquet("hf://datasets/LangAGI-Lab/medbullets/" + splits["train"])
test_original_df = pd.read_parquet("hf://datasets/LangAGI-Lab/medbullets/" + splits["test"])

In [ ]:
def get_full_url(short_url):
  try:
      response = requests.get(short_url, allow_redirects=True, timeout=10)

  except requests.exceptions.RequestException as e:
      print(f"An error occurred: {e}")
      return None

  return response.url

In [ ]:
# Get full link for train set
for i in range(len(train_original_df)):
  link = train_original_df.iloc[i]['link']

  if not (link.startswith('https://step2.medbullets.com')):
    link = get_full_url(link)

  train_original_df.loc[i, 'link'] = link

In [ ]:
# Get full link for test set
for i in range(len(test_original_df)):
  link = test_original_df.iloc[i]['link']

  if not (link.startswith('https://step2.medbullets.com')):
    link = get_full_url(link)

  test_original_df.loc[i, 'link'] = link

In [ ]:
train_original_df.to_csv("/content/train_original.csv")
test_original_df.to_csv("/content/test_original.csv")

In [ ]:
train_deduplicated_df = train_original_df.drop_duplicates(subset=['link'])
test_deduplicated_df = test_original_df.drop_duplicates(subset=['link'])

In [ ]:
train_deduplicated_df.to_csv("/content/train_deduplicated.csv")
test_deduplicated_df.to_csv("/content/test_deduplicated.csv")

In [ ]:
# Load filtered set
train_image_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/train_image.csv")
test_image_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/test_image.csv")

In [ ]:
train_placeholder_df = train_deduplicated_df[train_original_df['link'].isin(train_image_df['link'])]
test_placeholder_df = test_deduplicated_df[test_original_df['link'].isin(test_image_df['link'])]

/tmp/ipython-input-1863608945.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train_placeholder_df = train_deduplicated_df[train_original_df['link'].isin(train_image_df['link'])]
/tmp/ipython-input-1863608945.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  test_placeholder_df = test_deduplicated_df[test_original_df['link'].isin(test_image_df['link'])]


In [ ]:
print(train_placeholder_df.shape)
print(test_placeholder_df.shape)
print(train_image_df.shape)
print(test_image_df.shape)

(113, 10)
(47, 10)
(113, 6)
(47, 6)


In [ ]:
train_placeholder_df = pd.merge(train_placeholder_df, train_image_df[['link', 'image_base64']], on='link', how='left')
test_placeholder_df = pd.merge(test_placeholder_df, test_image_df[['link', 'image_base64']], on='link', how='left')

In [ ]:
train_placeholder_df.to_csv("/content/train_image_raw.csv")
test_placeholder_df.to_csv("/content/test_image_raw.csv")

In [ ]:
train_image_raw_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/train_image_raw.csv")
test_image_raw_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/test_image_raw.csv")

In [ ]:
train_image_rewritten_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/train_image_raw.csv")
test_image_rewritten_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/test_image_raw.csv")

# Synthesize data

In [ ]:
def split_text_to_tag(text, tags):
  parsed_data = {}

  tag_pattern = '|'.join(re.escape(tag) for tag in tags)
  split_regex = fr'(<\/?(?:{tag_pattern})>)'

  split_result = re.split(split_regex, text)

  cleaned_split_result = [item.strip() for item in split_result if item.strip()]

  current_tag = None

  for item in cleaned_split_result:
      if item.startswith('<') and item.endswith('>'):
          if item.startswith('</'): # Closing tag
              current_tag = None
          else: # Opening tag
              current_tag = item[1:-1] # Extract tag name without <>
              parsed_data[current_tag] = "" # Initialize with empty string, will be populated by next item
      elif current_tag:
          # Append to existing content, as there might be multiple lines of text between tags
          if parsed_data[current_tag]:
              parsed_data[current_tag] += "\n" + item.strip()
          else:
              parsed_data[current_tag] = item.strip()
  return parsed_data

In [ ]:
def get_question_rewriter_prompt(question, diagnostic_reasoning, choice_a, choice_b, choice_c, choice_d, choice_e, answer):
  question_rewriter_prompt = f"""
    You are an expert clinician–educator.
    Your job is to:
    - Rewrite the question and the corresponding reasoning from multiple-choice format to open-ended format, so that the question and the reasoning do not appear to be multiple-choice, but open-ended.
    - Ensure that the rewritten question and reasoning are accurate, based solely on the content given to you.
    - If there is any problem or concern that should be flagged, simply abort and output it.

    You are given the multiple choices, the question, the answer, and the reasoning.

    The question include an image. The image are passed to you together with the question and the reasoning.

    RULES (Read Carefully—No Exceptions)
    1. Source Fidelity – Include information from only the given content.
    • Do NOT invent, embellish, or “smooth out” missing data.

    2. Structure the Question
    • USMLE format.

    3. Structure of the reasoning
    • Initial reasoning -> Image grounding -> Choice elimination -> Summary

    4. Use the XML Tags Exactly as Shown
    • <question> . . . </question> – the question given to the student.
    • <diagnostic_reasoning> . . . </diagnostic_reasoning> – numbered bullet reasons, each built as a full sentence followed by a direct quote.

    5. What Goes Inside <question>
    • The rewritten question to open-ended format.
    • Keep as much content unchanged as possible, as long as that content does not appear to be multiple-choice.

    6. What Goes Inside <diagnostic_reasoning>
    • Rewritten the reasoning to open-ended format.
    • The steps to reach the answer from the question, each step as a paragraph or a full sentence as you wish.
    • Keep as much content unchanged as possible, as long as that content does not appear to be multiple-choice.
    • Focus solely on the reasoning explanation how to get the correct diagnosis, not the the advice or idea from other people mentioned.
    • Discard the summary and the incorrect answers.
    • UNDER NO CIRCUMSTANCE USING NUMBERED POINT.
    • UNDER NO CIRCUMSTANCE USING NUMBERED POINT.
    • UNDER NO CIRCUMSTANCE USING NUMBERED POINT.


    OUTPUT TEMPLATE (if normal) (copy exactly, especially the tag)
    <question>
    [Your rewritten question, faithful to the original question]
    </question>

    <diagnostic_reasoning>
    [Your rewritten reasoning, faithful to the original reasoning]
    </diagnostic_reasoning>

    OUTPUT TEMPLATE (if abnormal) (copy exactly, especially the tag)
    <flag>
    [Your description of the problem or concern]
    </flag>

    INPUT TEMPLATE
    <question>
    {question}
    </question>

    <diagnostic_reasoning>
    {diagnostic_reasoning}
    </diagnostic_reasoning>

    <choices>
    A: {choice_a}
    B: {choice_b}
    C: {choice_c}
    D: {choice_d}
    E: {choice_e}
    </choices>

    SUPPLIED ANSWER
    <answer>
    {answer}
    </answer>
  """

  return question_rewriter_prompt

In [ ]:
writer = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [ ]:
process_count = 0

# List to store the original indices of rows that failed processing
failed_indices = []

# Iterate through the original DataFrame's actual index labels
for original_idx in train_image_raw_df.index:
  # Access rows using .loc with the original index label
  current_raw_row = train_image_raw_df.loc[original_idx]
  current_rewritten_row = train_image_rewritten_df.loc[original_idx]

  if current_raw_row['link'] == current_rewritten_row['link']:
    link = current_raw_row['link']
    question_id = link.split("=")[1]

    print(f"Question {question_id} found")
  else:
    print(f"Question {question_id} not found for original_idx {original_idx} - marking for deletion.")
    print()
    failed_indices.append(original_idx)
    continue

  question = current_raw_row['question']
  diagnostic_reasoning = current_raw_row['explanation']
  choice_a = current_raw_row['opa']
  choice_b = current_raw_row['opb']
  choice_c = current_raw_row['opc']
  choice_d = current_raw_row['opd']
  choice_e = current_raw_row['ope']
  answer = current_raw_row['answer']
  image_base64 = current_raw_row['image_base64']

  rewritten_prompt = get_question_rewriter_prompt(question, diagnostic_reasoning, choice_a, choice_b, choice_c, choice_d, choice_e, answer)
  rewritten_response = writer.models.generate_content(
    model=MODEL_2, contents=[rewritten_prompt, types.Part.from_bytes(data=image_base64,mime_type='image/jpeg')], config=CONFIG
  )

  if rewritten_response.text is None:
    print(f"Question {question_id} not processed due to some reason - marking for deletion.")
    print()
    failed_indices.append(original_idx)
    continue

  rewritten_data = split_text_to_tag(rewritten_response.text, ['question', 'diagnostic_reasoning'])

  train_image_rewritten_df.loc[original_idx, 'question'] = rewritten_data['question']
  train_image_rewritten_df.loc[original_idx, 'explanation'] = rewritten_data['diagnostic_reasoning']

  process_count += 1
  print(f"Process {process_count} questions")
  print()
  time.sleep(6)

# After the loop, drop all failed rows from train_image_rewritten_df at once
if failed_indices:
  train_image_rewritten_df = train_image_rewritten_df.drop(index=failed_indices)
  print(f"Removed {len(failed_indices)} rows from train_image_rewritten_df due to processing failures.")

In [ ]:
process_count = 0

# List to store the original indices of rows that failed processing
failed_indices = []

# Iterate through the original DataFrame's actual index labels
for original_idx in test_image_raw_df.index:
  # Access rows using .loc with the original index label
  current_raw_row = test_image_raw_df.loc[original_idx]
  current_rewritten_row = test_image_rewritten_df.loc[original_idx]

  if current_raw_row['link'] == current_rewritten_row['link']:
    link = current_raw_row['link']
    question_id = link.split("=")[1]

    print(f"Question {question_id} found")
  else:
    print(f"Question {question_id} not found for original_idx {original_idx} - marking for deletion.")
    failed_indices.append(original_idx)
    continue

  question = current_raw_row['question']
  diagnostic_reasoning = current_raw_row['explanation']
  choice_a = current_raw_row['opa']
  choice_b = current_raw_row['opb']
  choice_c = current_raw_row['opc']
  choice_d = current_raw_row['opd']
  choice_e = current_raw_row['ope']
  answer = current_raw_row['answer']
  image_base64 = current_raw_row['image_base64']

  rewritten_prompt = get_question_rewriter_prompt(question, diagnostic_reasoning, choice_a, choice_b, choice_c, choice_d, choice_e, answer)
  rewritten_response = writer.models.generate_content(
    model=MODEL_2, contents=[rewritten_prompt, types.Part.from_bytes(data=image_base64,mime_type='image/jpeg')], config=CONFIG
  )

  if rewritten_response.text is None:
    print(f"Question {question_id} not processed due to some reason - marking for deletion.")
    failed_indices.append(original_idx)
    continue

  rewritten_data = split_text_to_tag(rewritten_response.text, ['question', 'diagnostic_reasoning'])

  test_image_rewritten_df.loc[original_idx, 'question'] = rewritten_data['question']
  test_image_rewritten_df.loc[original_idx, 'explanation'] = rewritten_data['diagnostic_reasoning']

  process_count += 1
  print(f"Process {process_count} questions")
  print()
  time.sleep(5)

# After the loop, drop all failed rows from train_image_rewritten_df at once
if failed_indices:
  test_image_rewritten_df = test_image_rewritten_df.drop(index=failed_indices)
  print(f"Removed {len(failed_indices)} rows from train_image_rewritten_df due to processing failures.")

Question 210697 found
Process 1 questions

Question 216625 found
Process 2 questions

Question 108912 found
Process 3 questions

Question 216495 found
Process 4 questions

Question 108726 found
Process 5 questions

Question 109464 found
Process 6 questions

Question 109964 found
Process 7 questions

Question 109292 found
Process 8 questions

Question 216637 found
Process 9 questions

Question 217652 found
Process 10 questions

Question 109255 found
Process 11 questions

Question 108992 found
Process 12 questions

Question 109244 found
Process 13 questions

Question 109944 found
Process 14 questions

Question 210369 found
Process 15 questions

Question 109208 found
Process 16 questions

Question 108974 found
Process 17 questions

Question 216521 found
Process 18 questions

Question 108933 found
Process 19 questions

Question 215176 found
Process 20 questions

Question 109699 found
Process 21 questions

Question 109249 found
Process 22 questions

Question 109433 found
Process 23 question

In [ ]:
train_image_rewritten_df.to_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/train_image_rewritten.csv")
test_image_rewritten_df.to_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/test_image_rewritten.csv")

# Qualitative exploration

In [ ]:
train_image_rewritten_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/train_image_rewritten.csv")
test_image_rewritten_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/test_image_rewritten.csv")

In [ ]:
train_image_raw_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/train_image_raw.csv")
test_image_raw_df = pd.read_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/test_image_raw.csv")

In [ ]:
train_image_raw_df.iloc[1]['question']

'A 27-year-old woman presents for her first prenatal visit at an estimated gestational age of 16 weeks and 4 days by last menstrual period. She is presenting late to prenatal care due to significant anxiety about her pregnancy, as she and her husband had struggled with infertility for several years prior to conceiving. She has had nausea and vomiting for about 2 months, tender and swollen breasts, and cravings for foods she typically does not eat. She took a pregnancy test about 10 weeks ago and was too nervous to read the result immediately. After 24 hours, she saw that it was positive. The patient has a past medical history of bulimia nervosa but has not been symptomatic in 2 years. She also had an appendectomy at age 15 for appendicitis. She is a Ph.D. student and her husband is a physician. Her temperature is 98.5°F (36.9°C), pulse is 75/min, blood pressure is 122/76 mmHg, and respirations are 13/min. The patient’s affect is normal and she is pleasant and excited about her pregnanc

In [ ]:
train_image_rewritten_df.iloc[1]['question']

'A 27-year-old woman presents for her first prenatal visit at an estimated gestational age of 16 weeks and 4 days by last menstrual period. She is presenting late to prenatal care due to significant anxiety about her pregnancy, as she and her husband had struggled with infertility for several years prior to conceiving. She reports experiencing nausea and vomiting for about 2 months, tender and swollen breasts, and cravings for foods she typically does not eat. She took a pregnancy test about 10 weeks ago and was too nervous to read the result immediately, and after 24 hours, she saw that it was positive. The patient has a past medical history of bulimia nervosa but has not been symptomatic in 2 years, and also had an appendectomy at age 15 for appendicitis. She is a Ph.D. student and her husband is a physician. Her vital signs are: temperature 98.5°F (36.9°C), pulse 75/min, blood pressure 122/76 mmHg, and respirations 13/min. The patient’s affect is normal and she is pleasant and excit

In [ ]:
train_image_raw_df.iloc[0]['explanation']

'This patient presents with a sudden, severe headache and a head CT showing bleeding in the subarachnoid space, which confirms a diagnosis of subarachnoid hemorrhage. The bleeding in a subarachnoid hemorrhage occurs between the arachnoid and pia mater.\n\nSubarachnoid hemorrhages present with a sudden, severe headache referred to as a “thunderclap” headache. Patients typically complain that the headache is the "worst headache of their life." Some patients also present with symptoms of meningeal irritation such as nausea and vomiting, nuchal rigidity, or focal neurologic deficits. Although most cases of subarachnoid hemorrhage are secondary to trauma, patients with risk factors such as hypertension and cocaine use can have non-traumatic presentations. Patients with an expanding bleed or acute neurologic changes require surgical clipping or embolization of the bleeding vessels.\n\nMacdonald and Schweizer review the evidence regarding the treatment of patients with subarachnoid hemorrhage

In [ ]:
train_image_rewritten_df.iloc[1]['explanation']

'The patient presents with signs and symptoms of pregnancy but a nongravid uterus on ultrasound and a negative pregnancy test. Therefore, the most likely diagnosis is pseudocyesis.\n\nPseudocyesis, or false pregnancy, most commonly presents with abdominal distension, breast tenderness, nausea, and other typical symptoms of pregnancy. Patients truly believe they are pregnant and may report positive pregnancy tests at home (perhaps due to false positives from waiting too long before reading the result) but office testing is negative. Ultrasound will be normal and show the absence of a developing fetus. The mainstay of treatment is explaining the diagnosis in a therapeutic manner and offering counseling to help patients cope.\n\nSmall reviews the evidence regarding the diagnosis and treatment of pseudocyesis, discussing how this disease is a heterogenous entity involving neuroendocrine changes and recommending clearly defining the cause of this disease in patients.\n\nFigure A shows a tra

In [ ]:
train_error = train_image_rewritten_df[train_image_rewritten_df['explanation'].str.contains(".\n2. ")]

In [ ]:
len(train_error)

0

In [ ]:
process_count = 0

# List to store the original indices of rows that failed processing
failed_indices = []

# Iterate through the original DataFrame's actual index labels
for original_idx in train_error.index:
  # Access rows using .loc with the original index label
  wrong_row = train_error.loc[original_idx]

  link = wrong_row['link']
  question_id = link.split("=")[1]

  print(f"Question {question_id} found")

  current_raw_row = train_image_raw_df[train_image_raw_df['link'] == link].iloc[0]

  question = current_raw_row['question']
  diagnostic_reasoning = current_raw_row['explanation']
  choice_a = current_raw_row['opa']
  choice_b = current_raw_row['opb']
  choice_c = current_raw_row['opc']
  choice_d = current_raw_row['opd']
  choice_e = current_raw_row['ope']
  answer = current_raw_row['answer']
  image_base64 = current_raw_row['image_base64']

  rewritten_prompt = get_question_rewriter_prompt(question, diagnostic_reasoning, choice_a, choice_b, choice_c, choice_d, choice_e, answer)
  rewritten_response = writer.models.generate_content(
    model=MODEL_2, contents=[rewritten_prompt, types.Part.from_bytes(data=image_base64,mime_type='image/jpeg')], config=CONFIG
  )

  if rewritten_response.text is None:
    print(f"Question {question_id} not processed due to some reason - marking for deletion.")
    print()
    failed_indices.append(original_idx)
    continue

  rewritten_data = split_text_to_tag(rewritten_response.text, ['question', 'diagnostic_reasoning'])

  train_image_rewritten_df.loc[original_idx, 'question'] = rewritten_data['question']
  train_image_rewritten_df.loc[original_idx, 'explanation'] = rewritten_data['diagnostic_reasoning']

  process_count += 1
  print(f"Process {process_count} questions")
  print()
  time.sleep(6)

# After the loop, drop all failed rows from train_image_rewritten_df at once
if failed_indices:
  train_error = train_error.drop(index=failed_indices)
  print(f"Removed {len(failed_indices)} rows from train_image_rewritten_df due to processing failures.")

Question 210369 found
Process 1 questions

Question 104911 found
Process 2 questions

Question 106349 found
Process 3 questions

Question 210076 found
Process 4 questions

Question 109958 found
Process 5 questions

Question 109919 found
Process 6 questions

Question 109777 found
Process 7 questions

Question 109944 found
Process 8 questions

Question 217243 found
Process 9 questions



In [ ]:
test_error = test_image_rewritten_df[test_image_rewritten_df['explanation'].str.contains(".\n2. ")]

In [ ]:
len(test_error)

0

In [ ]:
process_count = 0

# List to store the original indices of rows that failed processing
failed_indices = []

# Iterate through the original DataFrame's actual index labels
for original_idx in test_error.index:
  # Access rows using .loc with the original index label
  wrong_row = test_error.loc[original_idx]

  link = wrong_row['link']
  question_id = link.split("=")[1]

  print(f"Question {question_id} found")

  current_raw_row = test_image_raw_df[test_image_raw_df['link'] == link].iloc[0]

  question = current_raw_row['question']
  diagnostic_reasoning = current_raw_row['explanation']
  choice_a = current_raw_row['opa']
  choice_b = current_raw_row['opb']
  choice_c = current_raw_row['opc']
  choice_d = current_raw_row['opd']
  choice_e = current_raw_row['ope']
  answer = current_raw_row['answer']
  image_base64 = current_raw_row['image_base64']

  rewritten_prompt = get_question_rewriter_prompt(question, diagnostic_reasoning, choice_a, choice_b, choice_c, choice_d, choice_e, answer)
  rewritten_response = writer.models.generate_content(
    model=MODEL_2, contents=[rewritten_prompt, types.Part.from_bytes(data=image_base64,mime_type='image/jpeg')], config=CONFIG
  )

  if rewritten_response.text is None:
    print(f"Question {question_id} not processed due to some reason - marking for deletion.")
    print()
    failed_indices.append(original_idx)
    continue

  rewritten_data = split_text_to_tag(rewritten_response.text, ['question', 'diagnostic_reasoning'])

  test_image_rewritten_df.loc[original_idx, 'question'] = rewritten_data['question']
  test_image_rewritten_df.loc[original_idx, 'explanation'] = rewritten_data['diagnostic_reasoning']

  process_count += 1
  print(f"Process {process_count} questions")
  print()
  time.sleep(6)

# After the loop, drop all failed rows from train_image_rewritten_df at once
if failed_indices:
  test_error = test_error.drop(index=failed_indices)
  print(f"Removed {len(failed_indices)} rows from train_image_rewritten_df due to processing failures.")

Question 109944 found
Process 1 questions

Question 210369 found
Process 2 questions

Question 210076 found
Process 3 questions

Question 214944 found
Process 4 questions

Question 109919 found
Process 5 questions



In [ ]:
train_image_rewritten_df.to_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/train_image_rewritten_fixed.csv")
test_image_rewritten_df.to_csv("/content/drive/MyDrive/Project_Medical_LMM/MedBullet/test_image_rewritten_fixed.csv")